# 05b — LightGBM V2

LightGBM trained on the V2 feature set with a two-phase Optuna hyperparameter search.

**Differences vs `05_model_lightgbm.ipynb`:**
- Uses `DATASET_FILE_V2` and `FEATURES_V2` (richer lag set, neighbor context, no redundant features)
- Hyperparameter search with **Optuna** instead of RandomizedSearchCV:
  - Phase 1 — wide search (broad ranges, ~60 trials) to find the best region
  - Phase 2 — narrow search (±30% around Phase 1 best, ~40 trials) to find the optimum
- Both phases run on a **stratified 500k-row sample** so they finish in minutes
- Final model retrained on the full training set

**V1 reference results (05_model_lightgbm.ipynb):**
- Best CV RMSE (search sample): 0.1755
- Test RMSE: 0.1632 | MAE: 0.1235 | R²: 0.6571

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import shap
import optuna
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error

from src.utils.config import (
    PATHS, FEATURES_V2, TARGET, SPLIT_DATE, RANDOM_STATE, DATASET_FILE_V2
)
from src.models.evaluate import (
    regression_metrics, plot_predictions, plot_residuals,
    plot_feature_importance, plot_error_by_group
)

optuna.logging.set_verbosity(optuna.logging.WARNING)
print('Libraries loaded OK')

---
## 1. Load and split

In [ ]:
df = pd.read_parquet(DATASET_FILE_V2)
df['datetime_hour'] = pd.to_datetime(df['datetime_hour'])

train = df[df['datetime_hour'] < SPLIT_DATE]
test  = df[df['datetime_hour'] >= SPLIT_DATE]

X_train, y_train = train[FEATURES_V2], train[TARGET]
X_test,  y_test  = test[FEATURES_V2],  test[TARGET]

avg_capacity = float(df['capacity'].drop_duplicates().mean())
print(f'Train: {len(train):,}  Test: {len(test):,}')
print(f'Features: {len(FEATURES_V2)}')

---
## 2. Stratified sample for hyperparameter search (~500k rows)

Searching over the full ~16M train rows would take many hours. A 500k sample
stratified by `horizon_hours` preserves the distribution and gives Optuna
enough signal to find good hyperparameters in minutes.

In [ ]:
SEARCH_SAMPLE = 500_000

train_sample = (
    train
    .groupby('horizon_hours', group_keys=False)
    .apply(lambda g: g.sample(
        n=min(len(g), SEARCH_SAMPLE // train['horizon_hours'].nunique()),
        random_state=RANDOM_STATE
    ))
)
train_sample = train_sample.sort_values('datetime_hour').reset_index(drop=True)

X_search = train_sample[FEATURES_V2]
y_search = train_sample[TARGET]
print(f'Search sample: {len(train_sample):,} rows ({len(train_sample)/len(train)*100:.1f}% of train)')

---
## 3. Optuna objective

Shared objective function used by both phases. The search space is passed in
as `space` — a dict mapping param name to `(low, high)` or `(choices,)` tuples.
This lets Phase 2 narrow the ranges without duplicating the objective logic.

In [ ]:
TSCV = TimeSeriesSplit(n_splits=3, gap=168)

def make_objective(X, y, space):
    """
    Returns an Optuna objective function that:
    - Samples hyperparameters from `space`
    - Runs 3-fold TimeSeriesSplit CV on (X, y)
    - Returns mean RMSE across folds
    
    `space` format:
      'param': ('int',   low, high)          -> trial.suggest_int
      'param': ('float', low, high, log=bool) -> trial.suggest_float
      'param': ('cat',   [choices])           -> trial.suggest_categorical
    """
    def objective(trial):
        params = {}
        for name, spec in space.items():
            kind = spec[0]
            if kind == 'int':
                params[name] = trial.suggest_int(name, spec[1], spec[2])
            elif kind == 'float':
                log = spec[3] if len(spec) > 3 else False
                params[name] = trial.suggest_float(name, spec[1], spec[2], log=log)
            elif kind == 'cat':
                params[name] = trial.suggest_categorical(name, spec[1])

        model = lgb.LGBMRegressor(
            **params,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1,
        )

        rmses = []
        for train_idx, val_idx in TSCV.split(X):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            model.fit(X_tr, y_tr)
            preds = model.predict(X_val)
            rmses.append(float(np.sqrt(mean_squared_error(y_val, preds))))

        return float(np.mean(rmses))

    return objective

print('Objective function defined.')

---
## 4. Phase 1 — Wide search

Broad ranges to identify the best region of the hyperparameter space.
60 trials × 3 folds = 180 LightGBM fits on the 500k sample.

Reference from V1 RandomSearch best: `num_leaves=95, n_estimators=500,
max_depth=12, lr=0.08, subsample=0.7, colsample=0.7, min_child=100,
reg_alpha=0.1, reg_lambda=0.1`

In [ ]:
PHASE1_TRIALS = 60

# Wide search space — covers a broad area including and beyond the V1 best params.
# V2 features are richer so the model may benefit from higher capacity (num_leaves)
# and stronger regularization.
phase1_space = {
    'n_estimators':      ('int',   200,  1000),
    'num_leaves':        ('int',   31,   255),
    'max_depth':         ('cat',   [-1, 8, 10, 12, 16, 20]),
    'learning_rate':     ('float', 0.02, 0.15, True),   # log scale: finer near 0
    'subsample':         ('float', 0.5,  1.0),
    'colsample_bytree':  ('float', 0.5,  1.0),
    'min_child_samples': ('int',   20,   200),
    'reg_alpha':         ('float', 1e-4, 0.5, True),
    'reg_lambda':        ('float', 1e-4, 0.5, True),
}

study1 = optuna.create_study(direction='minimize',
                              sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study1.optimize(make_objective(X_search, y_search, phase1_space),
                n_trials=PHASE1_TRIALS,
                show_progress_bar=True)

p1_best = study1.best_params
p1_rmse = study1.best_value
print(f'\nPhase 1 best CV RMSE: {p1_rmse:.4f}')
print(f'Phase 1 best params:')
for k, v in p1_best.items():
    print(f'  {k}: {v}')

In [ ]:
# Visualize Phase 1 optimization history
trials_p1 = study1.trials_dataframe()

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(trials_p1['number'], trials_p1['value'], alpha=0.5, s=20, color='steelblue', label='Trial RMSE')
running_min = trials_p1['value'].cummin()
ax.plot(trials_p1['number'], running_min, color='crimson', linewidth=2, label='Best so far')
ax.set_xlabel('Trial')
ax.set_ylabel('CV RMSE')
ax.set_title('Phase 1 — Wide search optimization history')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/05b_phase1_history.png', dpi=120)
plt.show()
print(f'Phase 1: {PHASE1_TRIALS} trials, best RMSE {p1_rmse:.4f}')

---
## 5. Phase 2 — Narrow search

Search ±30% around Phase 1 best values. For categorical params (`max_depth`)
we keep only the best value ± 1 step.

Phase 2 starts a fresh study so Optuna doesn't inherit Phase 1's prior. This
avoids the sampler over-weighting regions that were well-explored in Phase 1.

In [ ]:
PHASE2_TRIALS = 40

def narrow(low_orig, high_orig, best, margin=0.30, min_width_pct=0.05):
    """Return a narrowed (low, high) range centered on `best` with ±margin."""
    span = high_orig - low_orig
    delta = max(span * margin, span * min_width_pct)
    return max(low_orig, best - delta), min(high_orig, best + delta)

def narrow_int(low_orig, high_orig, best, margin=0.30):
    lo, hi = narrow(low_orig, high_orig, best, margin)
    return int(round(lo)), int(round(hi))

# Build Phase 2 space dynamically from Phase 1 results
phase2_space = {
    'n_estimators': ('int',
                     *narrow_int(200, 1000, p1_best['n_estimators'])),
    'num_leaves':   ('int',
                     *narrow_int(31, 255, p1_best['num_leaves'])),
    'max_depth':    ('cat', [p1_best['max_depth']]),   # fix to Phase 1 winner
    'learning_rate': ('float',
                      *narrow(0.02, 0.15, p1_best['learning_rate']), True),
    'subsample':     ('float',
                      *narrow(0.5, 1.0, p1_best['subsample'])),
    'colsample_bytree': ('float',
                         *narrow(0.5, 1.0, p1_best['colsample_bytree'])),
    'min_child_samples': ('int',
                          *narrow_int(20, 200, p1_best['min_child_samples'])),
    'reg_alpha':  ('float',
                   *narrow(1e-4, 0.5, p1_best['reg_alpha']), True),
    'reg_lambda': ('float',
                   *narrow(1e-4, 0.5, p1_best['reg_lambda']), True),
}

print('Phase 2 search ranges:')
for k, v in phase2_space.items():
    if v[0] == 'float':
        print(f'  {k}: [{v[1]:.4f}, {v[2]:.4f}]  (Phase 1 best: {p1_best[k]:.4f})')
    elif v[0] == 'int':
        print(f'  {k}: [{v[1]}, {v[2]}]  (Phase 1 best: {p1_best[k]})')
    else:
        print(f'  {k}: {v[1]}  (fixed)')

In [ ]:
study2 = optuna.create_study(direction='minimize',
                              sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE + 1))
study2.optimize(make_objective(X_search, y_search, phase2_space),
                n_trials=PHASE2_TRIALS,
                show_progress_bar=True)

p2_best = study2.best_params
p2_rmse = study2.best_value
print(f'\nPhase 2 best CV RMSE: {p2_rmse:.4f}')
print(f'Phase 2 best params:')
for k, v in p2_best.items():
    print(f'  {k}: {v}')

In [ ]:
# Summary comparison: V1 baseline vs Phase 1 vs Phase 2
V1_SEARCH_RMSE = 0.1755  # from 05_model_lightgbm.ipynb

print('=== Hyperparameter search summary ===')
print(f'  V1 RandomSearch best CV RMSE : {V1_SEARCH_RMSE:.4f}')
print(f'  V2 Phase 1 best  CV RMSE     : {p1_rmse:.4f}  (delta {p1_rmse - V1_SEARCH_RMSE:+.4f})')
print(f'  V2 Phase 2 best  CV RMSE     : {p2_rmse:.4f}  (delta {p2_rmse - V1_SEARCH_RMSE:+.4f})')

# Visualize both phases together
trials_p2 = study2.trials_dataframe()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, trials, title, color in zip(
    axes,
    [trials_p1, trials_p2],
    ['Phase 1 — Wide', 'Phase 2 — Narrow'],
    ['steelblue', 'seagreen']
):
    ax.scatter(trials['number'], trials['value'], alpha=0.5, s=20, color=color)
    ax.plot(trials['number'], trials['value'].cummin(), color='crimson', linewidth=2)
    ax.set_xlabel('Trial')
    ax.set_ylabel('CV RMSE')
    ax.set_title(title)
plt.suptitle('Optuna two-phase search — optimization history', y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/05b_optuna_history.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 6. Retrain on full training set with Phase 2 best params

In [ ]:
best_lgbm_v2 = lgb.LGBMRegressor(
    **p2_best,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)
best_lgbm_v2.fit(X_train, y_train)
print('Retrained on full training set with Phase 2 best params.')

---
## 7. Evaluate

In [ ]:
y_pred_train = best_lgbm_v2.predict(X_train)
y_pred_test  = best_lgbm_v2.predict(X_test)

print('=== LIGHTGBM V2 ===')
m_train = regression_metrics(y_train, y_pred_train, 'Train', avg_capacity)
print()
m_test  = regression_metrics(y_test,  y_pred_test,  'Test',  avg_capacity)

print()
print('=== vs V1 (05_model_lightgbm.ipynb) ===')
V1_TEST_RMSE = 0.1632
V1_TEST_MAE  = 0.1235
V1_TEST_R2   = 0.6571
print(f'  RMSE: V1={V1_TEST_RMSE:.4f}  V2={m_test["rmse"]:.4f}  delta={m_test["rmse"]-V1_TEST_RMSE:+.4f}')
print(f'  MAE:  V1={V1_TEST_MAE:.4f}  V2={m_test["mae"]:.4f}  delta={m_test["mae"]-V1_TEST_MAE:+.4f}')
print(f'  R²:   V1={V1_TEST_R2:.4f}  V2={m_test["r2"]:.4f}  delta={m_test["r2"]-V1_TEST_R2:+.4f}')

---
## 8. RMSE by horizon

In [ ]:
# V1 reference values from 05_model_lightgbm.ipynb (re-enter if available)
V1_RMSE_BY_HORIZON = {}  # fill with {1: 0.xx, 3: 0.xx, ...} if you want comparison lines

rows = []
for h in sorted(test['horizon_hours'].unique()):
    mask = test['horizon_hours'] == h
    rmse = float(np.sqrt(mean_squared_error(y_test[mask], y_pred_test[mask])))
    rows.append({'horizon_hours': h, 'rmse_v2': rmse})

df_h = pd.DataFrame(rows)
print(df_h.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_h['horizon_hours'], df_h['rmse_v2'], marker='o', color='seagreen', label='LightGBM V2')
if V1_RMSE_BY_HORIZON:
    v1_h = sorted(V1_RMSE_BY_HORIZON.keys())
    ax.plot(v1_h, [V1_RMSE_BY_HORIZON[h] for h in v1_h],
            marker='s', color='steelblue', linestyle='--', label='LightGBM V1')
ax.set_xlabel('Horizon (hours ahead)')
ax.set_ylabel('RMSE')
ax.set_title('LightGBM V2 — RMSE by prediction horizon')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/05b_lgbm_v2_rmse_by_horizon.png', dpi=150)
plt.show()

---
## 9. Error analysis by distance to beach

In [ ]:
test_eval = test.copy()
test_eval['y_pred'] = y_pred_test
test_eval['sq_err'] = (test_eval[TARGET] - test_eval['y_pred']) ** 2

test_eval['dist_group'] = pd.cut(
    test_eval['dist_beach'],
    bins=[0, 0.5, 1.5, 3, 100],
    labels=['<500m', '500m-1.5km', '1.5-3km', '>3km']
)

rmse_by_dist = (
    test_eval.groupby('dist_group')['sq_err']
    .mean().apply(np.sqrt)
    .reset_index()
    .rename(columns={'sq_err': 'rmse', 'dist_group': 'group'})
)

fig = plot_error_by_group(rmse_by_dist.rename(columns={'group': 'dist_group'}),
                          group_col='dist_group',
                          title='LightGBM V2 — RMSE by distance to beach')
fig.savefig('../reports/figures/05b_lgbm_v2_rmse_by_dist.png', dpi=150)
plt.show()

---
## 10. SHAP — global feature importance

In [ ]:
sample_idx = X_test.sample(5000, random_state=RANDOM_STATE).index
X_sample   = X_test.loc[sample_idx]

explainer   = shap.TreeExplainer(best_lgbm_v2)
shap_values = explainer.shap_values(X_sample)

shap.summary_plot(shap_values, X_sample, show=False)
plt.tight_layout()
plt.savefig('../reports/figures/05b_lgbm_v2_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Mean absolute SHAP — bar chart sorted by importance
mean_shap = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=FEATURES_V2
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 8))
mean_shap.plot(kind='barh', ax=ax, color='seagreen')
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('LightGBM V2 — Feature importance (mean |SHAP|)')
plt.tight_layout()
plt.savefig('../reports/figures/05b_lgbm_v2_shap_bar.png', dpi=150)
plt.show()

In [ ]:
# Single prediction waterfall
row_idx = 0
shap.waterfall_plot(
    shap.Explanation(values=shap_values[row_idx],
                     base_values=explainer.expected_value,
                     data=X_sample.iloc[row_idx],
                     feature_names=FEATURES_V2),
    show=False
)
plt.tight_layout()
plt.savefig('../reports/figures/05b_lgbm_v2_shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 11. Prediction and residual plots

In [ ]:
fig = plot_predictions(y_test.values, y_pred_test,
                       rmse=m_test['rmse'], r2=m_test['r2'],
                       title='LightGBM V2 — Predicted vs Actual')
fig.savefig('../reports/figures/05b_lgbm_v2_predictions.png', dpi=150)
plt.show()

fig = plot_residuals(y_test.values, y_pred_test, title='LightGBM V2 — Residuals')
fig.savefig('../reports/figures/05b_lgbm_v2_residuals.png', dpi=150)
plt.show()

---
## 12. Save model

In [ ]:
model_path = PATHS['models'] / 'lightgbm_v2.joblib'
joblib.dump(best_lgbm_v2, model_path)
print(f'Model saved to {model_path}')
print(f'Final params: {p2_best}')